# Real-IAD 데이터 분석 + 증강·학습 과제 · OpenCV

**이번 실습 범위**
1. 클래스별 밝기: 품목 × 불량 유형 × 카메라별로 비교
2. GAN 참고용 불량 위치 히트맵: train의 불량 마스크로 생성
3. 불량 마스크 → 연결 성분 → 바운딩박스 → CSV·COCO·YOLO 라벨
4. 모델 입력 전처리: **1024 / 512 / 256** RGB·마스크·작은 불량 보존 비교
5. **학생 과제:** 밝기 및 OpenCV 증강을 구현하고 실제 학습으로 효과 평가

1~4는 실행 가능한 분석 코드, 5는 단계·힌트·빈 함수·결과표만 제공합니다.
GAN 모델 구현·증강 정답·학습 정답은 제공하지 않습니다.
기본 실행은 **합성 데모**이며 실제 Real-IAD 통계나 모델 성능이 아닙니다.
학습 데이터 전체를 메모리에 올리지 않으며, 기본 분석에는 GPU·LLM·API 키가 필요 없습니다.
`analysis_guide_legacy.ipynb`는 이전 일반 분석 흐름을 보존한 참고본입니다.

## 0. 설치·경로·실험 조건

압축을 풀고 이 노트북과 core.py가 있는 폴더에서 실행합니다.
터미널에서 `python -m pip install -r requirements-notebook.txt` 후 `python -m jupyter lab`.
실제 데이터는 `USE_DEMO=False`, ROOT·JSON_DIR·CATEGORIES를 변경하세요.
OpenCV 증강 과제는 두 버전 모두 requirements-assignment.txt가 추가로 필요합니다.

**클래스 정의:** category는 제품 품목(toothbrush/toy), anomaly_class는 제품의 불량 코드(OK/HS/ZW 등)입니다.
불량 제품의 모든 시점에 결함이 보이는 것은 아닙니다. 밝기 그래프의 클래스는 제품 수준 라벨입니다.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Image as DisplayImage

from core import (
    build_manifest, audit, project_split, analyze,
    read_rgb, read_mask, nearest, gray
)
from coursework import (
    run_focused, mask_to_boxes,
    coarse_occupancy, show_boxes
)
from backend import NAME

%matplotlib inline

# ==========================================
# 사용자 설정: 데이터셋 경로만 변경
# ==========================================

DATA_ROOT = Path(r"../../dataset")

SEED = 42
SIZES = (1024, 512, 256)
EXPECTED_VIEWS = ("C1", "C2", "C3", "C4", "C5")

CATEGORIES = None
MAX_PRODUCTS = 12

SPLITS = ("train", "val", "validation")
USE_SUPERVISED_PROJECT_SPLIT = False
HOLDOUT_DEFECTS = ()

OUTPUT = Path("notebook_results_real")
OUTPUT.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"데이터셋 경로가 존재하지 않습니다: {DATA_ROOT}"
    )

print("Backend:", NAME)
print("Dataset:", DATA_ROOT.resolve())
print("Input sizes:", SIZES)

In [ ]:
import re
from collections import Counter

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def detect_view(path):
    """파일명 또는 폴더명에서 C1~C5 추출"""
    for part in reversed(path.parts):
        match = re.search(
            r"(?<![A-Za-z0-9])C([1-5])(?![0-9])",
            part,
            re.IGNORECASE
        )
        if match:
            return f"C{match.group(1)}"
    return None


def detect_split(path):
    for part in path.parts:
        if part.lower() in {
            "train", "val", "validation", "test"
        }:
            return part.lower()
    return None


def detect_label(path):
    parts = [p.lower() for p in path.parts]

    if "good" in parts or "normal" in parts:
        return "normal"

    if "anomaly" in parts or "defect" in parts or "ng" in parts:
        return "anomaly"

    return "unknown"


def discover_dataset(root):
    """전체 데이터셋 재귀 탐색"""

    images = []
    json_files = []

    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue

        if path.suffix.lower() in IMAGE_EXTS:
            images.append(path)

        elif path.suffix.lower() == ".json":
            json_files.append(path)

    print("Images:", len(images))
    print("JSON files:", len(json_files))

    return images, json_files


def create_metadata(root, output):
    images, json_files = discover_dataset(root)

    records = []

    for path in images:
        relative = path.relative_to(root)
        parts = relative.parts

        # 최상위 하위 폴더를 임시 카테고리로 사용
        category = parts[0] if len(parts) > 1 else "unknown"

        records.append({
            "category": category,
            "filename": path.name,
            "image_path": str(path.resolve()),
            "relative_path": str(relative),
            "view": detect_view(relative),
            "split": detect_split(relative),
            "label": detect_label(relative),
            "extension": path.suffix.lower(),
            "parent_folder": path.parent.name
        })

    df = pd.DataFrame(
        records,
        columns=[
            "category", "filename", "image_path",
            "relative_path", "view", "split",
            "label", "extension", "parent_folder"
        ]
    )

    csv_path = output / "metadata.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print("\nMetadata saved:", csv_path)
    print("\nCategory distribution:")
    print(df["category"].value_counts())

    print("\nView distribution:")
    print(df["view"].value_counts(dropna=False))

    print("\nSplit distribution:")
    print(df["split"].value_counts(dropna=False))

    print("\nLabel distribution:")
    print(df["label"].value_counts(dropna=False))

    return df, json_files


metadata, json_files = create_metadata(DATA_ROOT, OUTPUT)

display(metadata.head(10))

In [ ]:
from pathlib import Path

# 실제 마스크가 들어 있는 상위 폴더 지정
MASK_ROOT = Path(r"../../dataset")

print("경로 존재:", MASK_ROOT.exists())
print("폴더 여부:", MASK_ROOT.is_dir())

mask_files = sorted(
    p for p in MASK_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() == ".png"
)

print("PNG 파일 수:", len(mask_files))

for p in mask_files[:10]:
    print(p.relative_to(MASK_ROOT))

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import re

METADATA_CSV = Path("notebook_results_real/metadata.csv")
OUTPUT = Path("notebook_results_real")
OUTPUT.mkdir(parents=True, exist_ok=True)

metadata = pd.read_csv(METADATA_CSV, dtype=str).fillna("")

if "image_id" not in metadata.columns:
    metadata["image_id"] = [
        f"IMG_{i:07d}" for i in range(len(metadata))
    ]

if metadata["image_id"].duplicated().any():
    raise ValueError("image_id 중복이 존재합니다.")

def normalize_name(path):
    name = Path(path).stem.lower()
    return re.sub(
        r"(_mask|_gt|_ground_truth)$", "", name
    )

# 원본 이미지 인덱스
image_index = defaultdict(list)

for row in metadata.to_dict("records"):
    key = (
        row["category"].strip().lower(),
        normalize_name(row["image_path"])
    )
    image_index[key].append(row)

matched = []
unmatched = []
ambiguous = []

for mask_path in mask_files:
    relative = mask_path.relative_to(MASK_ROOT)

    # 카테고리 이름이 경로 중간에 있어도 탐지
    categories = {
        str(c).lower()
        for c in metadata["category"].unique()
    }

    category_matches = [
        part.lower() for part in relative.parts[:-1]
        if part.lower() in categories
    ]

    if not category_matches:
        unmatched.append({
            "mask_path": str(mask_path),
            "reason": "category_not_found"
        })
        continue

    category = category_matches[0]

    key = (category, normalize_name(mask_path))
    candidates = image_index.get(key, [])

    if len(candidates) == 1:
        image = candidates[0]

        matched.append({
            "image_id": image["image_id"],
            "category": category,
            "image_path": image["image_path"],
            "mask_path": str(mask_path.resolve()),
            "view": image.get("view", ""),
            "match_method": "category_stem"
        })

    elif len(candidates) > 1:
        ambiguous.append({
            "mask_path": str(mask_path),
            "category": category,
            "candidate_count": len(candidates)
        })

    else:
        unmatched.append({
            "mask_path": str(mask_path),
            "category": category,
            "reason": "filename_not_found"
        })

matched_df = pd.DataFrame(
    matched,
    columns=[
        "image_id", "category", "image_path",
        "mask_path", "view", "match_method"
    ]
)

matched_df.to_csv(
    OUTPUT / "mask_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(unmatched).to_csv(
    OUTPUT / "unmatched_masks.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(ambiguous).to_csv(
    OUTPUT / "ambiguous_masks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("원본 이미지:", len(metadata))
print("마스크 PNG:", len(mask_files))
print("연결 성공:", len(matched_df))
print("연결 실패:", len(unmatched))
print("중복 후보:", len(ambiguous))

display(matched_df.head(10))

In [ ]:
from pathlib import Path
import pandas as pd

# 첫 번째 마스크를 기준으로 후보 확인
mask_path = mask_files[0]
relative = mask_path.relative_to(MASK_ROOT)

categories = set(metadata["category"].str.lower())

category = next(
    (p.lower() for p in relative.parts[:-1]
     if p.lower() in categories),
    None
)

key = (category, normalize_name(mask_path))
candidates = image_index.get(key, [])

print("===== 마스크 =====")
print("경로:", mask_path)
print("카테고리:", category)
print("파일명:", mask_path.name)
print("정규화 이름:", key[1])

print("\n===== 원본 후보 =====")
print("후보 개수:", len(candidates))

candidate_df = pd.DataFrame(candidates)

columns = [
    c for c in [
        "image_id", "category", "filename",
        "image_path", "relative_path",
        "view", "split", "label"
    ]
    if c in candidate_df.columns
]

display(candidate_df[columns].head(30))

In [ ]:
from collections import Counter

candidate_counts = Counter()
examples = []

categories = set(metadata["category"].str.lower())

for mask_path in mask_files:
    relative = mask_path.relative_to(MASK_ROOT)

    category = next(
        (p.lower() for p in relative.parts[:-1]
         if p.lower() in categories),
        None
    )

    key = (category, normalize_name(mask_path))
    candidates = image_index.get(key, [])

    candidate_counts[len(candidates)] += 1

    if len(candidates) > 1 and len(examples) < 10:
        examples.append({
            "mask_path": str(mask_path),
            "candidate_count": len(candidates),
            "original_paths": [
                c["image_path"]
                for c in candidates[:5]
            ]
        })

print("===== 후보 개수별 마스크 분포 =====")

display(pd.DataFrame(
    sorted(candidate_counts.items()),
    columns=["원본 후보 수", "마스크 수"]
))

print("\n===== 중복 사례 =====")

for example in examples[:5]:
    print("\n마스크:", example["mask_path"])
    print("후보:", example["candidate_count"])

    for path in example["original_paths"]:
        print("  JPG:", path)

In [ ]:
from pathlib import Path
import pandas as pd

# ==========================================
# 1. 경로 설정
# ==========================================

METADATA_CSV = Path("notebook_results_real/metadata.csv")
OUTPUT = Path("notebook_results_real")
OUTPUT.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. 기존 CSV 읽기
# ==========================================

df = pd.read_csv(METADATA_CSV, dtype=str).fillna("")

# 컬럼명 공백 제거
df.columns = df.columns.str.strip()

# 필수 컬럼 검사
required = {"image_path", "relative_path"}

missing = required - set(df.columns)

if missing:
    raise ValueError(f"필수 컬럼 누락: {missing}")

# image_id가 없으면 자동 생성
if "image_id" not in df.columns:
    df.insert(
        0,
        "image_id",
        [f"IMG_{i:07d}" for i in range(len(df))]
    )

# image_id 중복 확인
if df["image_id"].duplicated().any():
    raise ValueError("image_id 중복이 존재합니다.")

df["extension_check"] = (
    df["image_path"]
    .map(lambda x: Path(x).suffix.lower())
)

print("전체 데이터:", len(df))
print("컬럼:", df.columns.tolist())

df["extension_check"] = (
    df["image_path"]
    .map(lambda x: Path(x).suffix.lower())
)

# ==========================================
# 3. JPG / PNG 분리
# ==========================================

images = df[
    df["extension_check"].isin([".jpg", ".jpeg"])
].copy()

masks = df[
    df["extension_check"] == ".png"
].copy()

print("전체:", len(df))
print("JPG 원본:", len(images))
print("PNG 마스크:", len(masks))

# ==========================================
# 4. 원본 이미지 경로 매칭
# ==========================================

# 파일명과 카테고리만 비교하지 않고
# 전체 상대 경로를 비교해 중복 방지

def get_match_key(row):
    relative = str(row["relative_path"])
    return str(Path(relative).with_suffix("")).replace(
        "\\", "/"
    ).lower()

images["match_key"] = images.apply(
    get_match_key, axis=1
)

masks["match_key"] = masks.apply(
    get_match_key, axis=1
)

# 원본 후보 중복 검사
duplicates = images[
    images["match_key"].duplicated(keep=False)
]

print("원본 중복 키:", len(duplicates))

if not duplicates.empty:
    raise ValueError(
        "동일 상대 경로의 JPG 원본이 중복되어 있습니다."
    )

# ==========================================
# 5. 마스크와 원본 연결
# ==========================================

image_lookup = images[
    ["image_id", "image_path", "match_key"]
].rename(columns={
    "image_id": "original_image_id",
    "image_path": "original_image_path"
})

mask_metadata = masks.merge(
    image_lookup,
    on="match_key",
    how="left",
    validate="many_to_one"
)

mask_metadata["matched"] = (
    mask_metadata["original_image_id"].notna()
    & mask_metadata["original_image_id"].ne("")
)

# ==========================================
# 6. CSV 저장
# ==========================================

images.drop(
    columns=["extension_check", "match_key"],
    errors="ignore"
).to_csv(
    OUTPUT / "image_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

mask_metadata.to_csv(
    OUTPUT / "mask_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

# 원본과 연결된 마스크만 추출
paired = mask_metadata[
    mask_metadata["matched"]
].copy()

paired.to_csv(
    OUTPUT / "image_mask_pairs.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================
# 7. 결과 확인
# ==========================================

print("\n===== 결과 =====")
print("원본 이미지:", len(images))
print("마스크 이미지:", len(masks))
print("연결 성공:", len(paired))
print(
    "연결 실패:",
    len(mask_metadata) - len(paired)
)

display(
    paired[
        [
            "category",
            "view",
            "original_image_path",
            "image_path"
        ]
    ].head(10)
)

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

samples = paired.sample(
    n=min(5, len(paired)),
    random_state=42
)

fig, axes = plt.subplots(
    len(samples), 3,
    figsize=(15, 4 * len(samples)),
    squeeze=False
)

for i, (_, row) in enumerate(samples.iterrows()):
    image = np.asarray(
        Image.open(row["original_image_path"]).convert("RGB")
    )

    mask_path = row.get("mask_path", row.get("image_path"))
    mask = np.asarray(
        Image.open(mask_path).convert("L")
    )

    if image.shape[:2] != mask.shape:
        print("크기 불일치:", row["original_image_path"])
        continue

    binary = mask > 0

    overlay = image.copy()
    overlay[binary] = (
        image[binary] * 0.5
        + np.array([255, 0, 0]) * 0.5
    ).astype(np.uint8)

    axes[i, 0].imshow(image)
    axes[i, 0].set_title(f"{row['category']} / {row['view']}")

    axes[i, 1].imshow(binary, cmap="gray")
    axes[i, 1].set_title("Defect Mask")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Overlay")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT = Path("notebook_results_real")

df = pd.read_csv(
    OUTPUT / "image_metadata.csv",
    dtype=str
).fillna("")

def parse_defect_info(relative_path):
    parts = str(relative_path).replace("\\", "/").split("/")

    upper_parts = [p.upper() for p in parts]

    if "NG" in upper_parts:
        idx = upper_parts.index("NG")

        defect_type = (
            parts[idx + 1]
            if idx + 1 < len(parts) - 1
            else "UNKNOWN"
        )

        return pd.Series({
            "quality": "NG",
            "defect_type": defect_type
        })

    if "OK" in upper_parts:
        return pd.Series({
            "quality": "OK",
            "defect_type": "NORMAL"
        })

    return pd.Series({
        "quality": "UNKNOWN",
        "defect_type": "UNKNOWN"
    })


df[["quality", "defect_type"]] = df[
    "relative_path"
].apply(parse_defect_info)

display(
    df[
        ["category", "view", "quality", "defect_type"]
    ].head(10)
)

In [ ]:
# 카테고리 + 불량 유형별 이미지 수
defect_summary = (
    df[df["quality"] == "NG"]
    .groupby(["category", "defect_type"])
    .agg(
        image_count=("image_path", "count"),
        view_count=("view", "nunique")
    )
    .reset_index()
    .sort_values(
        ["category", "image_count"],
        ascending=[True, False]
    )
)

display(defect_summary)

defect_summary.to_csv(
    OUTPUT / "defect_type_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 카테고리별 불량 유형 개수
category_summary = (
    defect_summary.groupby("category")
    .agg(
        defect_type_count=("defect_type", "nunique"),
        defect_image_count=("image_count", "sum")
    )
    .reset_index()
    .sort_values("defect_type_count", ascending=False)
)

display(category_summary)

In [ ]:
view_summary = pd.crosstab(
    index=[
        df["category"],
        df["defect_type"]
    ],
    columns=df["view"]
)

display(view_summary)

view_summary.to_csv(
    OUTPUT / "defect_view_summary.csv",
    encoding="utf-8-sig"
)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

ng_df = df[df["quality"] == "NG"]

samples = (
    ng_df.groupby(
        ["category", "defect_type"],
        group_keys=False
    )
    .sample(n=1, random_state=42)
)

# 한 번에 너무 많이 표시하지 않도록 제한
samples = samples.head(12)

fig, axes = plt.subplots(
    3, 4,
    figsize=(16, 12)
)

axes = axes.flatten()

for ax, (_, row) in zip(axes, samples.iterrows()):
    image = Image.open(row["image_path"]).convert("RGB")

    ax.imshow(image)
    ax.set_title(
        f"{row['category']} / {row['defect_type']}\n"
        f"View: {row['view']}"
    )
    ax.axis("off")

for ax in axes[len(samples):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. 클래스별 밝기 분석

**단계**: RGB 공통 디코딩 → gray=0.299R+0.587G+0.114B → 이미지별 mean·median·P05·P95 → 품목×불량 코드×뷰 집계.
한 영상의 픽셀을 모두 독립 표본으로 보지 말고 제품 수도 확인합니다.
히스토그램은 각 이미지를 합계 1로 정규화한 뒤 평균하므로 큰 이미지가 더 큰 가중치를 갖지 않습니다.

- 전체 영상의 밝기 차이는 배경·카메라·촬영 조건 영향도 포함합니다.
- mask 양성 내부(defect_mean)와 그 밖(nondefect_mean)도 비교합니다.
- 마스크 밖은 제품 정상 표면+배경이 섞인 영역이며 **순수 배경/정상 제품 정답**이 아닙니다.
- 밝기 차이가 있다는 이유만으로 그 값이 결함의 원인이라고 결론내리지 않습니다.

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT = Path("notebook_results_real")

images = pd.read_csv(
    OUTPUT / "image_metadata.csv",
    dtype=str
).fillna("")

pairs = pd.read_csv(
    OUTPUT / "image_mask_pairs.csv",
    dtype=str
).fillna("")

# 이전 단계에서 만든 불량 유형 정보가
# CSV에 저장되지 않았을 경우 경로에서 추출
def parse_info(path):
    parts = str(path).replace("\\", "/").split("/")
    upper = [p.upper() for p in parts]

    if "NG" in upper:
        i = upper.index("NG")
        defect = parts[i + 1] if i + 1 < len(parts) - 1 else "UNKNOWN"
        product = next(
            (p for p in parts[i + 2:] if p.upper().startswith("S")),
            "UNKNOWN"
        )
        return "NG", defect, product

    if "OK" in upper:
        i = upper.index("OK")
        product = next(
            (p for p in parts[i + 1:] if p.upper().startswith("S")),
            "UNKNOWN"
        )
        return "OK", "NORMAL", product

    return "UNKNOWN", "UNKNOWN", "UNKNOWN"


images[["quality", "defect_type", "product_id"]] = (
    images["relative_path"]
    .apply(lambda p: pd.Series(parse_info(p)))
)

# 제품 식별자는 카테고리·불량 유형까지 포함
images["product_key"] = (
    images["category"] + "/" +
    images["quality"] + "/" +
    images["defect_type"] + "/" +
    images["product_id"]
)

# 마스크 연결
mask_lookup = pairs[
    ["original_image_id", "image_path"]
].rename(columns={
    "original_image_id": "image_id",
    "image_path": "mask_path"
})

images = images.merge(
    mask_lookup,
    on="image_id",
    how="left",
    validate="one_to_one"
)

print("분석 이미지:", len(images))
print("마스크 연결:", images["mask_path"].notna().sum())
print("제품 수:", images["product_key"].nunique())

In [ ]:
def analyze_brightness(row):
    image_path = Path(row["image_path"])

    with Image.open(image_path) as im:
        rgb = np.asarray(im.convert("RGB"), dtype=np.float32)

    gray = (
        0.299 * rgb[:, :, 0]
        + 0.587 * rgb[:, :, 1]
        + 0.114 * rgb[:, :, 2]
    )

    values = gray.ravel()

    hist, _ = np.histogram(
        values,
        bins=256,
        range=(0, 256)
    )

    hist = hist.astype(np.float64)
    hist /= max(hist.sum(), 1)

    result = {
        "image_id": row["image_id"],
        "category": row["category"],
        "quality": row["quality"],
        "defect_type": row["defect_type"],
        "product_key": row["product_key"],
        "view": row["view"],
        "mean": float(values.mean()),
        "median": float(np.median(values)),
        "p05": float(np.percentile(values, 5)),
        "p95": float(np.percentile(values, 95)),
        "defect_mean": np.nan,
        "nondefect_mean": np.nan,
        "defect_pixels": 0,
        "histogram": hist
    }

    mask_path = row.get("mask_path")

    if pd.notna(mask_path) and str(mask_path).strip():
        with Image.open(mask_path) as im:
            mask = np.asarray(im.convert("L"))

        if mask.shape != gray.shape:
            raise ValueError(
                f"크기 불일치: {image_path.name}"
            )

        positive = mask > 0
        negative = ~positive

        result["defect_pixels"] = int(positive.sum())

        if positive.any():
            result["defect_mean"] = float(
                gray[positive].mean()
            )

        if negative.any():
            result["nondefect_mean"] = float(
                gray[negative].mean()
            )

    return result

In [ ]:
# 시험: 1000
# 전체: None
MAX_IMAGES = None

work = (
    images.head(MAX_IMAGES)
    if MAX_IMAGES is not None
    else images
)

records = []
histograms = []

for row in tqdm(
    work.to_dict("records"),
    total=len(work)
):
    result = analyze_brightness(row)

    histograms.append(result.pop("histogram"))
    records.append(result)

brightness_df = pd.DataFrame(records)

histograms = np.stack(histograms)

brightness_df.to_csv(
    OUTPUT / "brightness_per_image.csv",
    index=False,
    encoding="utf-8-sig"
)

np.save(
    OUTPUT / "brightness_histograms.npy",
    histograms
)

print("분석 완료:", len(brightness_df))
display(brightness_df.head())

In [ ]:
brightness_summary = (
    brightness_df
    .groupby(
        ["category", "defect_type", "view"],
        dropna=False
    )
    .agg(
        image_count=("image_id", "count"),
        product_count=("product_key", "nunique"),
        mean_brightness=("mean", "mean"),
        median_brightness=("median", "mean"),
        p05_brightness=("p05", "mean"),
        p95_brightness=("p95", "mean"),
        defect_mean=("defect_mean", "mean"),
        nondefect_mean=("nondefect_mean", "mean")
    )
    .reset_index()
)

brightness_summary["brightness_difference"] = (
    brightness_summary["defect_mean"]
    - brightness_summary["nondefect_mean"]
)

brightness_summary.to_csv(
    OUTPUT / "brightness_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(brightness_summary)

In [ ]:
category_brightness = (
    brightness_df
    .groupby(["category", "quality"])["mean"]
    .mean()
    .unstack()
)

category_brightness.plot(
    kind="bar",
    figsize=(16, 6)
)

plt.title("Mean Brightness by Category and Quality")
plt.ylabel("Mean grayscale intensity")
plt.xlabel("Category")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

### 밝기 분석 해석 과제

아래 표의 빈칸을 직접 작성하세요. 품목과 view를 고정한 비교를 최소 2개 포함합니다.
평균만 보지 말고 분포 겹침·표본 수·배경 영향·결함 부위의 상대 밝기를 근거로 적으세요.
밝기 증강 범위는 이 분석과 시각 검수로 정하고 test 통계로 정하지 않습니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT = Path("notebook_results_real")

df = pd.read_csv(
    OUTPUT / "brightness_per_image.csv"
)

histograms = np.load(
    OUTPUT / "brightness_histograms.npy"
)

summary = pd.read_csv(
    OUTPUT / "brightness_summary.csv"
)

assert len(df) == len(histograms), "CSV와 히스토그램 행 수 불일치"
assert histograms.shape[1] == 256, "히스토그램 bin 수 확인 필요"

print("분석 이미지:", len(df))
print("품목:", df["category"].nunique())
print("불량 유형:", df["defect_type"].nunique())
print("제품:", df["product_key"].nunique())

In [ ]:
# 정상 / 불량 구분
df["quality"] = df["quality"].astype(str).str.upper()

ok = df[df["quality"] == "OK"]
ng = df[df["quality"] == "NG"]

# 정상과 불량이 모두 존재하는 category × view
ok_keys = set(zip(ok["category"], ok["view"]))
ng_keys = set(zip(ng["category"], ng["view"]))

common_keys = sorted(ok_keys & ng_keys)

print("비교 가능한 품목 × View:", len(common_keys))
print(common_keys[:20])

In [ ]:
def compare_distributions(ok_values, ng_values):
    ok_values = np.asarray(ok_values, dtype=float)
    ng_values = np.asarray(ng_values, dtype=float)

    ok_values = ok_values[np.isfinite(ok_values)]
    ng_values = ng_values[np.isfinite(ng_values)]

    if len(ok_values) < 2 or len(ng_values) < 2:
        return {
            "mean_diff": np.nan,
            "overlap": np.nan,
            "cohens_d": np.nan
        }

    # 평균 차이
    mean_diff = ng_values.mean() - ok_values.mean()

    # 공통 0~256 구간
    bins = np.linspace(0, 256, 33)

    ok_hist, _ = np.histogram(ok_values, bins=bins)
    ng_hist, _ = np.histogram(ng_values, bins=bins)

    ok_prob = ok_hist / ok_hist.sum()
    ng_prob = ng_hist / ng_hist.sum()

    overlap = np.minimum(ok_prob, ng_prob).sum()

    # Cohen's d
    n1, n2 = len(ok_values), len(ng_values)

    pooled_var = (
        (n1 - 1) * ok_values.var(ddof=1)
        + (n2 - 1) * ng_values.var(ddof=1)
    ) / (n1 + n2 - 2)

    cohens_d = (
        mean_diff / np.sqrt(pooled_var)
        if pooled_var > 0 else np.nan
    )

    return {
        "mean_diff": mean_diff,
        "overlap": overlap,
        "cohens_d": cohens_d
    }

In [ ]:
records = []

for category, view in common_keys:

    ok_group = ok[
        (ok["category"] == category)
        & (ok["view"] == view)
    ]

    ng_subset = ng[
        (ng["category"] == category)
        & (ng["view"] == view)
    ]

    for defect_type, ng_group in ng_subset.groupby("defect_type"):

        metrics = compare_distributions(
            ok_group["mean"],
            ng_group["mean"]
        )

        relative_brightness = (
            ng_group["defect_mean"]
            - ng_group["nondefect_mean"]
        ).dropna()

        records.append({
            "category": category,
            "view": view,
            "defect_type": defect_type,

            "ok_images": len(ok_group),
            "ng_images": len(ng_group),

            "ok_products": ok_group["product_key"].nunique(),
            "ng_products": ng_group["product_key"].nunique(),

            "ok_mean": ok_group["mean"].mean(),
            "ng_mean": ng_group["mean"].mean(),

            "mean_diff": metrics["mean_diff"],
            "distribution_overlap": metrics["overlap"],
            "cohens_d": metrics["cohens_d"],

            "defect_relative_brightness": (
                relative_brightness.mean()
                if len(relative_brightness) else np.nan
            )
        })

comparison = pd.DataFrame(records)

comparison.to_csv(
    OUTPUT / "brightness_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    comparison.sort_values(
        ["category", "view", "defect_type"]
    ).head(30)
)

In [ ]:
eligible = comparison[
    (comparison["ok_images"] >= 5)
    & (comparison["ng_images"] >= 5)
].copy()

selected = eligible.sort_values(
    ["ng_products", "ok_products"],
    ascending=False
).head(2)

if len(selected) < 2:
    print("비교 가능한 조합이 2개 미만입니다.")
    print("데이터 범위 또는 MAX_IMAGES 설정을 확인하세요.")

for _, row in selected.iterrows():

    category = row["category"]
    view = row["view"]
    defect = row["defect_type"]

    ok_values = ok[
        (ok["category"] == category)
        & (ok["view"] == view)
    ]["mean"]

    ng_values = ng[
        (ng["category"] == category)
        & (ng["view"] == view)
        & (ng["defect_type"] == defect)
    ]["mean"]

    plt.figure(figsize=(10, 5))

    bins = np.linspace(0, 256, 33)

    plt.hist(
        ok_values,
        bins=bins,
        density=True,
        alpha=0.5,
        label=f"OK (n={len(ok_values)})"
    )

    plt.hist(
        ng_values,
        bins=bins,
        density=True,
        alpha=0.5,
        label=f"NG-{defect} (n={len(ng_values)})"
    )

    plt.title(f"{category} / {view} / OK vs NG-{defect}")
    plt.xlabel("Image mean brightness")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT = Path("notebook_results_real")

df = pd.read_csv(OUTPUT / "brightness_per_image.csv")

df["quality"] = df["quality"].astype(str).str.upper()

# 결함 부위와 마스크 외부의 상대 밝기
df["defect_contrast"] = (
    df["defect_mean"] - df["nondefect_mean"]
)

# 상대 밝기 차이의 절댓값
df["abs_defect_contrast"] = df["defect_contrast"].abs()

records = []

for (category, view), group in df.groupby(["category", "view"]):

    ok = group[group["quality"] == "OK"]
    ng = group[group["quality"] == "NG"]

    ok_mean = ok["mean"].mean()
    ok_std = ok["mean"].std()

    ok_cv = (
        ok_std / abs(ok_mean)
        if len(ok) >= 2 and pd.notna(ok_mean)
        and abs(ok_mean) > 1e-8
        else np.nan
    )

    records.append({
        "category": category,
        "view": view,
        "ok_images": len(ok),
        "ng_images": len(ng),
        "ok_products": ok["product_key"].nunique(),
        "ng_products": ng["product_key"].nunique(),
        "ok_mean": ok_mean,
        "ok_std": ok_std,
        "ok_cv": ok_cv,
        "ng_mean": ng["mean"].mean(),
        "mean_difference": ng["mean"].mean() - ok_mean,
        "defect_contrast": ng["defect_contrast"].mean(),
        "abs_defect_contrast": ng["abs_defect_contrast"].mean()
    })

diagnostics = pd.DataFrame(records)

diagnostics.to_csv(
    OUTPUT / "brightness_preprocessing_diagnostics.csv",
    index=False,
    encoding="utf-8-sig"
)

display(diagnostics)

## 4. GAN 참고용 불량 위치 히트맵

산점도 대신 **정규화 위치 격자(기본 64×64)의 영상별 점유율**을 만듭니다.
품목 × 불량 유형 × 카메라를 분리하며 **project_split=train만** 사용합니다.
양성 원본 픽셀을 격자에 매핑하므로 1픽셀 불량도 누락하지 않습니다. 픽셀 밀도나 GAN 생성 결과가 아닙니다.

- `occupancy_known`: 해당 train 그룹의 확인 가능한 마스크 수(빈 마스크 포함)로 나눈 점유율
- `occupancy_positive`: 양성 마스크 영상 수만으로 나눈 조건부 점유율
- 미제공·오류는 분모에서 제외하고 n_unknown으로 표시
- train에 양성 마스크가 0개면 positive 맵은 NA, 데이터가 없으면 그룹 자체가 없을 수 있음

**GAN 활용 힌트**: 소스 제품·유형·뷰에 맞는 위치 후보를 제안하는 참고 자료입니다.
제품 전경·부위 검수 없이 확률맵만 따라 배치하지 마세요. 5뷰 좌표를 섞거나 동일 물리 결함으로 연결하지 않습니다.
선택 표본으로 만든 히트맵이므로 실제 GAN 준비에서는 train 전체로 다시 산출하세요.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

OUTPUT = Path("notebook_results_real")

images = pd.read_csv(
    OUTPUT / "image_metadata.csv",
    dtype=str
).fillna("")

# ==========================================
# 1. 경로에서 제품 정보 추출
# ==========================================

def extract_product_info(path):
    parts = str(path).replace("\\", "/").split("/")
    upper = [p.upper() for p in parts]

    if "NG" in upper:
        idx = upper.index("NG")

        defect_type = (
            parts[idx + 1]
            if idx + 1 < len(parts) - 1
            else "UNKNOWN"
        )

        product_id = (
            parts[idx + 2]
            if idx + 2 < len(parts) - 1
            else "UNKNOWN"
        )

        return "NG", defect_type, product_id

    if "OK" in upper:
        idx = upper.index("OK")

        product_id = (
            parts[idx + 1]
            if idx + 1 < len(parts) - 1
            else "UNKNOWN"
        )

        return "OK", "NORMAL", product_id

    return "UNKNOWN", "UNKNOWN", "UNKNOWN"


images[["quality", "defect_type", "product_id"]] = (
    images["relative_path"]
    .apply(lambda x: pd.Series(extract_product_info(x)))
)

# ==========================================
# 2. 고유 제품 식별자 생성
# ==========================================

images["product_key"] = (
    images["category"] + "/" +
    images["quality"] + "/" +
    images["defect_type"] + "/" +
    images["product_id"]
)

# UNKNOWN 제품이 있다면 분할 중단
unknown = images[
    images["product_id"] == "UNKNOWN"
]

if len(unknown):
    raise ValueError(
        f"제품 ID를 추출하지 못한 이미지 {len(unknown)}개"
    )

# ==========================================
# 3. 제품 단위 분할
# ==========================================

products = images[
    ["product_key", "category", "quality", "defect_type"]
].drop_duplicates("product_key")

train_products, temp_products = train_test_split(
    products,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_products, test_products = train_test_split(
    temp_products,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

split_map = {}

for key in train_products["product_key"]:
    split_map[key] = "train"

for key in val_products["product_key"]:
    split_map[key] = "val"

for key in test_products["product_key"]:
    split_map[key] = "test"

images["project_split"] = (
    images["product_key"].map(split_map)
)

# ==========================================
# 4. 누수 검사
# ==========================================

leakage = (
    images.groupby("product_key")["project_split"]
    .nunique()
)

assert leakage.max() == 1, "제품 단위 데이터 누수 발생"

# ==========================================
# 5. 저장
# ==========================================

images.to_csv(
    OUTPUT / "image_metadata_with_split.csv",
    index=False,
    encoding="utf-8-sig"
)

print("===== 분할 결과 =====")

display(
    images.groupby("project_split").agg(
        image_count=("image_path", "count"),
        product_count=("product_key", "nunique")
    )
)

print("제품 단위 누수 검사 통과")

In [ ]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT = Path("notebook_results_real")
HEATMAP_DIR = OUTPUT / "gan_heatmaps"
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

GRID_SIZE = 64

images = pd.read_csv(
    OUTPUT / "image_metadata_with_split.csv",
    dtype=str
).fillna("")

pairs = pd.read_csv(
    OUTPUT / "image_mask_pairs.csv",
    dtype=str
).fillna("")

# 분할 정보는 명시적으로 확인
if "project_split" not in images.columns:
    raise ValueError(
        "project_split 컬럼이 없습니다. "
        "제품 단위 train/val/test 분할을 먼저 연결하세요."
    )

images["project_split"] = (
    images["project_split"].str.strip().str.lower()
)

train_images = images[
    images["project_split"] == "train"
].copy()

print("전체 원본 이미지:", len(images))
print("Train 이미지:", len(train_images))

In [ ]:
split_distribution = pd.crosstab(
    index=[
        images["category"],
        images["defect_type"]
    ],
    columns=images["project_split"]
)

display(split_distribution)

In [ ]:
def extract_defect_type(path):
    parts = str(path).replace("\\", "/").split("/")
    upper = [p.upper() for p in parts]

    if "NG" in upper:
        idx = upper.index("NG")
        if idx + 1 < len(parts) - 1:
            return parts[idx + 1]

    return "UNKNOWN"


train_images["defect_type"] = (
    train_images["relative_path"]
    .map(extract_defect_type)
)

# 이전 단계에서 생성한 pairs 컬럼 구조 사용
mask_col = (
    "mask_path"
    if "mask_path" in pairs.columns
    else "image_path"
)

mask_lookup = pairs[
    ["original_image_id", mask_col]
].rename(columns={
    "original_image_id": "image_id",
    mask_col: "mask_path"
})

if mask_lookup["image_id"].duplicated().any():
    raise ValueError("하나의 원본에 마스크가 여러 개 연결되어 있습니다.")

train_images = train_images.merge(
    mask_lookup,
    on="image_id",
    how="left",
    validate="one_to_one"
)

# 불량 이미지만 분석
train_ng = train_images[
    train_images["defect_type"] != "UNKNOWN"
].copy()

print("Train 불량 이미지:", len(train_ng))
print("마스크 연결:", train_ng["mask_path"].notna().sum())

In [ ]:
def mask_to_occupancy(mask, grid_size=64):
    """
    원본 마스크의 양성 픽셀을 정규화 격자로 매핑.

    반환:
        (grid_size, grid_size) bool 배열
    """

    mask = np.asarray(mask)

    if mask.ndim != 2:
        raise ValueError("마스크는 2차원 배열이어야 합니다.")

    h, w = mask.shape

    if h == 0 or w == 0:
        raise ValueError("빈 크기의 마스크입니다.")

    ys, xs = np.nonzero(mask > 0)

    occupancy = np.zeros(
        (grid_size, grid_size),
        dtype=bool
    )

    if len(xs) == 0:
        return occupancy

    grid_x = np.minimum(
        (xs.astype(np.int64) * grid_size) // w,
        grid_size - 1
    )

    grid_y = np.minimum(
        (ys.astype(np.int64) * grid_size) // h,
        grid_size - 1
    )

    occupancy[grid_y, grid_x] = True

    return occupancy

In [ ]:
def calculate_group_heatmap(group, grid_size=64):

    occupancy_sum = np.zeros(
        (grid_size, grid_size),
        dtype=np.int64
    )

    n_known = 0
    n_positive = 0
    n_empty = 0
    n_unknown = 0

    errors = []

    for row in group.itertuples(index=False):

        mask_path = getattr(row, "mask_path", "")

        if not mask_path or pd.isna(mask_path):
            n_unknown += 1
            errors.append({
                "image_id": row.image_id,
                "reason": "mask_missing"
            })
            continue

        try:
            with Image.open(mask_path) as im:
                mask = np.asarray(im.convert("L"))

            occupancy = mask_to_occupancy(
                mask,
                grid_size=grid_size
            )

        except Exception as e:
            n_unknown += 1
            errors.append({
                "image_id": row.image_id,
                "reason": str(e)
            })
            continue

        n_known += 1

        if occupancy.any():
            n_positive += 1
        else:
            n_empty += 1

        occupancy_sum += occupancy.astype(np.int64)

    occupancy_known = (
        occupancy_sum / n_known
        if n_known > 0
        else np.full((grid_size, grid_size), np.nan)
    )

    occupancy_positive = (
        occupancy_sum / n_positive
        if n_positive > 0
        else np.full((grid_size, grid_size), np.nan)
    )

    return {
        "occupancy_known": occupancy_known,
        "occupancy_positive": occupancy_positive,
        "n_known": n_known,
        "n_positive": n_positive,
        "n_empty": n_empty,
        "n_unknown": n_unknown,
        "errors": errors
    }

In [ ]:
summary_records = []
error_records = []

groups = train_ng.groupby(
    ["category", "defect_type", "view"],
    dropna=False
)

for (category, defect_type, view), group in tqdm(
    groups,
    total=groups.ngroups
):

    result = calculate_group_heatmap(
        group,
        grid_size=GRID_SIZE
    )

    group_dir = (
        HEATMAP_DIR
        / str(category)
        / str(defect_type)
        / str(view)
    )

    group_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    np.save(
        group_dir / "occupancy_known.npy",
        result["occupancy_known"]
    )

    np.save(
        group_dir / "occupancy_positive.npy",
        result["occupancy_positive"]
    )

    summary_records.append({
        "category": category,
        "defect_type": defect_type,
        "view": view,
        "n_images": len(group),
        "n_known": result["n_known"],
        "n_positive": result["n_positive"],
        "n_empty": result["n_empty"],
        "n_unknown": result["n_unknown"],
        "known_map_path": str(
            group_dir / "occupancy_known.npy"
        ),
        "positive_map_path": str(
            group_dir / "occupancy_positive.npy"
        )
    })

    for error in result["errors"]:
        error_records.append({
            "category": category,
            "defect_type": defect_type,
            "view": view,
            **error
        })

heatmap_summary = pd.DataFrame(summary_records)
heatmap_errors = pd.DataFrame(error_records)

heatmap_summary.to_csv(
    HEATMAP_DIR / "heatmap_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

heatmap_errors.to_csv(
    HEATMAP_DIR / "heatmap_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("생성된 그룹:", len(heatmap_summary))
print("마스크 오류:", len(heatmap_errors))

display(heatmap_summary.head(20))

In [ ]:
def show_heatmap(category, defect_type, view):

    group_dir = (
        HEATMAP_DIR
        / category
        / defect_type
        / view
    )

    known = np.load(
        group_dir / "occupancy_known.npy"
    )

    positive = np.load(
        group_dir / "occupancy_positive.npy"
    )

    fig, axes = plt.subplots(
        1, 2,
        figsize=(12, 5)
    )

    for ax, data, title in zip(
        axes,
        [known, positive],
        ["Occupancy Known", "Occupancy Positive"]
    ):
        im = ax.imshow(
            data,
            cmap="hot",
            vmin=0,
            vmax=1,
            origin="upper"
        )

        ax.set_title(title)
        ax.set_xlabel("Normalized X")
        ax.set_ylabel("Normalized Y")

        plt.colorbar(
            im,
            ax=ax,
            label="Image occupancy rate"
        )

    plt.suptitle(
        f"{category} / {defect_type} / {view}"
    )

    plt.tight_layout()
    plt.show()


# 실제 생성된 첫 번째 그룹 시각화
if not heatmap_summary.empty:
    first = heatmap_summary.iloc[0]

    show_heatmap(
        first["category"],
        first["defect_type"],
        first["view"]
    )

In [ ]:
assert (
    heatmap_summary["n_known"]
    == heatmap_summary["n_positive"]
    + heatmap_summary["n_empty"]
).all()

assert (
    heatmap_summary["n_images"]
    == heatmap_summary["n_known"]
    + heatmap_summary["n_unknown"]
).all()

print("마스크 집계 검증 통과")

In [ ]:
for path in HEATMAP_DIR.rglob("occupancy_*.npy"):

    arr = np.load(path)

    valid = arr[np.isfinite(arr)]

    assert ((valid >= 0) & (valid <= 1)).all(), path

print("히트맵 값 범위 검증 통과")

### GAN 위치 설계 과제 (생성기는 구현하지 않음)

1. train의 품목·유형·뷰별 양성 제품 수를 확인합니다.
2. 히트맵과 원본 5뷰·실제 결함 마스크를 대조합니다.
3. 합성 가능한 전경/부위 ROI를 검수해서 별도 정의합니다.
4. 위치 후보·마스크 크기·방향·강도의 표본화 방법을 설명합니다.
5. 편집 마스크와 실제 생성 불량 마스크의 차이, 박스 승인 방법을 정합니다.
6. source_sample_key, view_id, seed, generator_version, approved_mask_ref를 기록할 양식을 설계합니다.

**제출:** GAN 위치 설계표 + 허용·거절 위치 예시. 지금 단계에서는 GAN 학습·생성 함수를 요구하지 않습니다.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT = Path("notebook_results_real")
HEATMAP_DIR = OUTPUT / "gan_heatmaps"
DESIGN_DIR = OUTPUT / "gan_design"
DESIGN_DIR.mkdir(parents=True, exist_ok=True)

images = pd.read_csv(
    OUTPUT / "image_metadata_with_split.csv",
    dtype=str
).fillna("")

pairs = pd.read_csv(
    OUTPUT / "image_mask_pairs.csv",
    dtype=str
).fillna("")

mask_col = (
    "mask_path" if "mask_path" in pairs.columns
    else "image_path"
)

lookup = pairs[
    ["original_image_id", mask_col]
].rename(columns={
    "original_image_id": "image_id",
    mask_col: "mask_path"
})

if lookup["image_id"].duplicated().any():
    raise ValueError("중복된 원본 이미지-마스크 연결")

train = images[
    images["project_split"].str.lower() == "train"
].copy()

train = train.merge(
    lookup,
    on="image_id",
    how="left",
    validate="one_to_one"
)

train["quality"] = train["quality"].str.upper()

# 히트맵 생성 시 사용한 양성 마스크 정의와 동일하게 유지
def mask_status(path):
    if not isinstance(path, str) or not path.strip():
        return "unknown"

    try:
        with Image.open(path) as im:
            mask = np.asarray(im.convert("L"))
        return "positive" if np.any(mask > 0) else "empty"
    except Exception:
        return "unknown"

ng = train[train["quality"] == "NG"].copy()
ng["mask_status"] = ng["mask_path"].map(mask_status)

positive = ng[ng["mask_status"] == "positive"]

positive_summary = (
    positive.groupby(["category", "defect_type", "view"])
    .agg(
        positive_images=("image_id", "nunique"),
        positive_products=("product_key", "nunique")
    )
    .reset_index()
)

all_groups = (
    ng.groupby(["category", "defect_type", "view"])
    .agg(
        train_ng_images=("image_id", "nunique"),
        train_ng_products=("product_key", "nunique")
    )
    .reset_index()
)

positive_summary = all_groups.merge(
    positive_summary,
    on=["category", "defect_type", "view"],
    how="left"
)

positive_summary[
    ["positive_images", "positive_products"]
] = positive_summary[
    ["positive_images", "positive_products"]
].fillna(0).astype(int)

positive_summary.to_csv(
    DESIGN_DIR / "positive_product_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(positive_summary)

In [ ]:
GRID_SIZE = 64
TOP_K = 5

design_records = []

for row in positive_summary.itertuples(index=False):
    category = row.category
    defect = row.defect_type
    view = row.view

    group_dir = (
        HEATMAP_DIR / category / defect / view
    )

    known_path = group_dir / "occupancy_known.npy"
    positive_path = group_dir / "occupancy_positive.npy"

    if not known_path.exists() or not positive_path.exists():
        continue

    known = np.load(known_path)
    positive_map = np.load(positive_path)

    if known.shape != (GRID_SIZE, GRID_SIZE):
        raise ValueError(f"격자 크기 불일치: {known_path}")

    if row.positive_images == 0:
        continue

    if not np.isfinite(positive_map).any():
        continue

    # 높은 점유율 셀을 우선 후보로 선정
    candidates = np.argwhere(
        np.isfinite(positive_map) & (positive_map > 0)
    )

    ranked = sorted(
        candidates,
        key=lambda p: positive_map[p[0], p[1]],
        reverse=True
    )[:TOP_K]

    for rank, (gy, gx) in enumerate(ranked, start=1):
        design_records.append({
            "category": category,
            "defect_type": defect,
            "view": view,
            "candidate_rank": rank,
            "grid_x": int(gx),
            "grid_y": int(gy),
            "center_x_norm": (gx + 0.5) / GRID_SIZE,
            "center_y_norm": (gy + 0.5) / GRID_SIZE,
            "occupancy_known": float(known[gy, gx]),
            "occupancy_positive": float(positive_map[gy, gx]),
            "positive_products": row.positive_products,
            "roi_status": "PENDING",
            "roi_reason": "",
            "reviewer": "",
            "review_notes": ""
        })

location_design = pd.DataFrame(design_records)

location_design.to_csv(
    DESIGN_DIR / "gan_location_design.csv",
    index=False,
    encoding="utf-8-sig"
)

display(location_design.head(20))

In [ ]:
def review_location(category, defect_type, view, sample_n=3):
    subset = positive[
        (positive["category"] == category)
        & (positive["defect_type"] == defect_type)
        & (positive["view"] == view)
    ]

    if subset.empty:
        print("해당 그룹에 양성 마스크가 없습니다.")
        return

    subset = subset.sample(
        min(sample_n, len(subset)),
        random_state=42
    )

    heatmap_path = (
        HEATMAP_DIR / category / defect_type / view
        / "occupancy_positive.npy"
    )

    heatmap = np.load(heatmap_path)

    fig, axes = plt.subplots(
        len(subset), 3,
        figsize=(14, 4 * len(subset)),
        squeeze=False
    )

    for i, row in enumerate(subset.itertuples(index=False)):
        with Image.open(row.image_path) as im:
            rgb = np.asarray(im.convert("RGB"))

        with Image.open(row.mask_path) as im:
            mask = np.asarray(im.convert("L")) > 0

        if rgb.shape[:2] != mask.shape:
            raise ValueError("원본과 마스크 해상도 불일치")

        overlay = rgb.copy()
        overlay[mask] = (
            0.5 * rgb[mask]
            + 0.5 * np.array([255, 0, 0])
        ).astype(np.uint8)

        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title("Original")

        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title("Actual Defect Mask")

        axes[i, 2].imshow(rgb)
        axes[i, 2].imshow(
            heatmap,
            extent=(0, rgb.shape[1], rgb.shape[0], 0),
            cmap="hot",
            alpha=0.45,
            vmin=0,
            vmax=1
        )
        axes[i, 2].set_title("Location Prior")

        for ax in axes[i]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()


# 실제 존재하는 그룹 선택
if not location_design.empty:
    sample_group = location_design.iloc[0]

    review_location(
        sample_group["category"],
        sample_group["defect_type"],
        sample_group["view"]
    )

In [ ]:
roi_review = location_design[
    [
        "category", "defect_type", "view",
        "candidate_rank",
        "grid_x", "grid_y",
        "center_x_norm", "center_y_norm",
        "roi_status", "roi_reason",
        "reviewer", "review_notes"
    ]
].copy()

roi_review["foreground_checked"] = ""
roi_review["part_checked"] = ""
roi_review["boundary_checked"] = ""
roi_review["source_image_ref"] = ""
roi_review["actual_mask_ref"] = ""
roi_review["reviewed_at"] = ""

roi_review.to_csv(
    DESIGN_DIR / "gan_roi_review.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
sampling_policy = pd.DataFrame([
    {
        "parameter": "location",
        "method": "approved_roi_weighted_sampling",
        "source": "occupancy_positive",
        "constraint": "approved foreground ROI only"
    },
    {
        "parameter": "size",
        "method": "empirical_area_distribution",
        "source": "train defect masks",
        "constraint": "category/type/view specific"
    },
    {
        "parameter": "orientation",
        "method": "empirical_orientation_distribution",
        "source": "train mask geometry",
        "constraint": "surface structure compatible"
    },
    {
        "parameter": "intensity",
        "method": "empirical_local_contrast",
        "source": "train defect brightness/color",
        "constraint": "visual review required"
    }
])

sampling_policy.to_csv(
    DESIGN_DIR / "gan_sampling_policy.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
generation_columns = [
    "generation_id",
    "source_sample_key",
    "view_id",
    "category",
    "defect_type",
    "seed",
    "generator_version",
    "location_policy_version",
    "roi_ref",
    "edit_mask_ref",
    "generated_image_ref",
    "generated_defect_mask_ref",
    "approved_mask_ref",
    "bbox_ref",
    "approval_status",
    "rejection_reason",
    "reviewer",
    "created_at",
    "reviewed_at"
]

generation_log = pd.DataFrame(
    columns=generation_columns
)

generation_log.to_csv(
    DESIGN_DIR / "gan_generation_log_template.csv",
    index=False,
    encoding="utf-8-sig"
)

print("GAN 생성 이력 양식 생성 완료")

## 5. 마스크 → 바운딩박스 추출

**단계**: 검증한 바이너리 마스크 → 8방향 연결 성분 → 각 성분의 최소 박스 → 원본 좌표 저장.
미세 결함 보존을 위해 기본 면적 제거·closing·팽창·박스 padding을 하지 않습니다.
분리된 성분은 각각 박스이며 하나의 물리적 결함이 여러 성분으로 나뉠 수도 있습니다.

좌표는 `[x0,y0,x1,y1]`, 오른쪽·아래 끝점 제외입니다.
예: 마스크가 x=100..119,y=200..209이면 `[100,200,120,210]`.
COCO는 `[x0,y0,w,h]`, YOLO는 `class cx/W cy/H w/W h/H`.
원본 불량 코드를 성분별 정답으로 추정하지 않고 **defect 1클래스**로 내보냅니다.

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

OUTPUT = Path("notebook_results_real")
BBOX_DIR = OUTPUT / "bbox_results"
BBOX_DIR.mkdir(parents=True, exist_ok=True)

pairs = pd.read_csv(
    OUTPUT / "image_mask_pairs.csv",
    dtype=str
).fillna("")

# 이전 단계에서 컬럼명이 다를 수 있으므로 대응
if "mask_path" not in pairs.columns:
    pairs = pairs.rename(columns={
        "image_path": "mask_path"
    })

required = {
    "original_image_id",
    "original_image_path",
    "mask_path"
}

missing = required - set(pairs.columns)

if missing:
    raise ValueError(f"필수 컬럼 누락: {missing}")

print("마스크 연결 이미지:", len(pairs))

In [ ]:
def extract_bboxes(mask):
    """
    입력:
        mask: 2D 바이너리 마스크 (0/1 또는 0/255)

    반환:
        성분별 bbox 목록
        좌표: [x0, y0, x1, y1]
        x1, y1은 exclusive
    """

    mask = np.asarray(mask)

    if mask.ndim != 2:
        raise ValueError("마스크는 2차원이어야 합니다.")

    unique_values = np.unique(mask)

    if not np.isin(unique_values, [0, 1, 255]).all():
        raise ValueError(
            f"바이너리 마스크가 아닙니다: {unique_values[:20]}"
        )

    binary = (mask > 0).astype(np.uint8)

    n_labels, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            binary,
            connectivity=8
        )
    )

    boxes = []

    # label=0은 배경
    for label_id in range(1, n_labels):

        x = int(stats[label_id, cv2.CC_STAT_LEFT])
        y = int(stats[label_id, cv2.CC_STAT_TOP])
        w = int(stats[label_id, cv2.CC_STAT_WIDTH])
        h = int(stats[label_id, cv2.CC_STAT_HEIGHT])
        area = int(stats[label_id, cv2.CC_STAT_AREA])

        boxes.append({
            "component_id": label_id,
            "x0": x,
            "y0": y,
            "x1": x + w,
            "y1": y + h,
            "width": w,
            "height": h,
            "area_pixels": area
        })

    return boxes

In [ ]:
# 처음에는 100장으로 시험
# 전체 실행하려면 None
MAX_IMAGES = 100

work = (
    pairs.head(MAX_IMAGES)
    if MAX_IMAGES is not None
    else pairs
)

bbox_records = []
error_records = []
image_records = []

for row in tqdm(
    work.itertuples(index=False),
    total=len(work)
):

    try:
        with Image.open(row.original_image_path) as im:
            W, H = im.size

        with Image.open(row.mask_path) as im:
            mask = np.asarray(im)

        if mask.ndim == 3:
            raise ValueError(
                "다채널 마스크: 바이너리 변환 규칙 확인 필요"
            )

        if mask.shape != (H, W):
            raise ValueError(
                f"크기 불일치: image={(H,W)}, mask={mask.shape}"
            )

        boxes = extract_bboxes(mask)

        image_records.append({
            "image_id": row.original_image_id,
            "image_path": row.original_image_path,
            "width": W,
            "height": H,
            "component_count": len(boxes)
        })

        for box in boxes:
            bbox_records.append({
                "image_id": row.original_image_id,
                "image_path": row.original_image_path,
                "mask_path": row.mask_path,
                "category": getattr(row, "category", ""),
                "view": getattr(row, "view", ""),
                "class_id": 0,
                "class_name": "defect",
                **box
            })

    except Exception as e:
        error_records.append({
            "image_id": row.original_image_id,
            "image_path": row.original_image_path,
            "mask_path": row.mask_path,
            "error": str(e)
        })

bbox_df = pd.DataFrame(bbox_records)
image_df = pd.DataFrame(image_records)
error_df = pd.DataFrame(error_records)

bbox_df.to_csv(
    BBOX_DIR / "defect_bboxes.csv",
    index=False,
    encoding="utf-8-sig"
)

image_df.to_csv(
    BBOX_DIR / "image_component_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

error_df.to_csv(
    BBOX_DIR / "bbox_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("처리 이미지:", len(image_df))
print("추출 박스:", len(bbox_df))
print("오류:", len(error_df))

display(bbox_df.head(10))

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

sample = pairs.iloc[0]

with Image.open(sample["mask_path"]) as im:
    print("Mask mode:", im.mode)
    mask = np.asarray(im)

print("Shape:", mask.shape)
print("Dtype:", mask.dtype)
print("Min:", mask.min())
print("Max:", mask.max())

values, counts = np.unique(mask, return_counts=True)

pixel_distribution = sorted(
    zip(values.tolist(), counts.tolist()),
    key=lambda x: x[1],
    reverse=True
)

print("\n가장 많은 픽셀 값 20개:")
for value, count in pixel_distribution[:20]:
    print(value, count)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(mask, cmap="gray")
axes[0].set_title("Original Mask")

axes[1].hist(
    mask.ravel(),
    bins=256,
    range=(0, 256)
)
axes[1].set_yscale("log")
axes[1].set_title("Mask Pixel Distribution")
axes[1].set_xlabel("Pixel Value")

plt.tight_layout()
plt.show()

In [ ]:
thresholds = [0, 32, 64, 127, 192, 254]

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 9)
)

for ax, threshold in zip(axes.flat, thresholds):

    binary = mask > threshold

    ax.imshow(binary, cmap="gray")
    ax.set_title(
        f"Threshold > {threshold}\n"
        f"Positive pixels: {binary.sum():,}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
with Image.open(sample["original_image_path"]) as im:
    rgb = np.asarray(im.convert("RGB"))

if mask.ndim != 2:
    raise ValueError(
        "다채널 마스크입니다. 채널 의미를 먼저 확인하세요."
    )

if rgb.shape[:2] != mask.shape:
    raise ValueError("원본과 마스크 크기가 다릅니다.")

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 10)
)

for ax, threshold in zip(axes.flat, thresholds):

    binary = mask > threshold

    overlay = rgb.copy()
    overlay[binary] = (
        0.5 * rgb[binary]
        + 0.5 * np.array([255, 0, 0])
    ).astype(np.uint8)

    ax.imshow(overlay)
    ax.set_title(f"Threshold > {threshold}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 예시: 검증 후 선택한 임계값을 입력
MASK_THRESHOLD = 127

binary_mask = (
    mask > MASK_THRESHOLD
).astype(np.uint8)

boxes = extract_bboxes(binary_mask)

print("추출된 Bounding Box:", len(boxes))

In [ ]:
def to_coco(box):
    return [
        int(box["x0"]),
        int(box["y0"]),
        int(box["width"]),
        int(box["height"])
    ]


def to_yolo(box, W, H):
    x0, y0 = box["x0"], box["y0"]
    x1, y1 = box["x1"], box["y1"]

    w = x1 - x0
    h = y1 - y0

    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2

    return [
        0,
        cx / W,
        cy / H,
        w / W,
        h / H
    ]

In [ ]:
coco = {
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "defect"}
    ]
}

image_id_map = {}

for i, row in enumerate(
    image_df.itertuples(index=False),
    start=1
):
    image_id_map[row.image_id] = i

    coco["images"].append({
        "id": i,
        "file_name": row.image_path,
        "width": int(row.width),
        "height": int(row.height)
    })

for annotation_id, row in enumerate(
    bbox_df.itertuples(index=False),
    start=1
):
    coco["annotations"].append({
        "id": annotation_id,
        "image_id": image_id_map[row.image_id],
        "category_id": 1,
        "bbox": [
            int(row.x0),
            int(row.y0),
            int(row.width),
            int(row.height)
        ],
        "area": int(row.area_pixels),
        "iscrowd": 0
    })

with open(
    BBOX_DIR / "defect_coco.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(coco, f, ensure_ascii=False, indent=2)

print("COCO annotations:", len(coco["annotations"]))

In [ ]:
YOLO_DIR = BBOX_DIR / "yolo_labels"
YOLO_DIR.mkdir(parents=True, exist_ok=True)

for row in image_df.itertuples(index=False):

    image_boxes = bbox_df[
        bbox_df["image_id"] == row.image_id
    ]

    lines = []

    for box in image_boxes.to_dict("records"):
        values = to_yolo(
            box,
            row.width,
            row.height
        )

        lines.append(
            f"{values[0]} "
            + " ".join(
                f"{v:.8f}" for v in values[1:]
            )
        )

    label_path = YOLO_DIR / f"{row.image_id}.txt"

    label_path.write_text(
        "\n".join(lines),
        encoding="utf-8"
    )

print("YOLO 라벨 생성 완료")

In [ ]:
import matplotlib.patches as patches

def show_bbox_sample(image_id):

    sample = image_df[
        image_df["image_id"] == image_id
    ].iloc[0]

    boxes = bbox_df[
        bbox_df["image_id"] == image_id
    ]

    with Image.open(sample["image_path"]) as im:
        image = np.asarray(im.convert("RGB"))

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)

    for row in boxes.itertuples(index=False):
        rect = patches.Rectangle(
            (row.x0, row.y0),
            row.width,
            row.height,
            linewidth=1.5,
            edgecolor="red",
            facecolor="none"
        )
        ax.add_patch(rect)

    ax.set_title(
        f"{image_id} | Components: {len(boxes)}"
    )
    ax.axis("off")
    plt.show()


if not image_df.empty:
    show_bbox_sample(image_df.iloc[0]["image_id"])

In [ ]:
# 1. 직사각형 영역
mask = np.zeros((300, 300), dtype=np.uint8)
mask[200:210, 100:120] = 255

boxes = extract_bboxes(mask)

assert len(boxes) == 1
assert [
    boxes[0]["x0"],
    boxes[0]["y0"],
    boxes[0]["x1"],
    boxes[0]["y1"]
] == [100, 200, 120, 210]

# 2. 단일 픽셀 결함
mask = np.zeros((10, 10), dtype=np.uint8)
mask[5, 7] = 255

boxes = extract_bboxes(mask)

assert len(boxes) == 1
assert [
    boxes[0]["x0"],
    boxes[0]["y0"],
    boxes[0]["x1"],
    boxes[0]["y1"]
] == [7, 5, 8, 6]

# 3. 대각선 연결: 8방향에서는 하나의 성분
mask = np.zeros((10, 10), dtype=np.uint8)
mask[2, 2] = 255
mask[3, 3] = 255

boxes = extract_bboxes(mask)

assert len(boxes) == 1

print("좌표·단일 픽셀·8방향 연결 테스트 통과")

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

sample = pairs.iloc[0]

with Image.open(sample["original_image_path"]) as im:
    rgb = np.asarray(im.convert("RGB"))

with Image.open(sample["mask_path"]) as im:
    print("Mask mode:", im.mode)
    mask = np.asarray(im)

if mask.ndim != 2:
    raise ValueError("다채널 마스크입니다. 채널 규칙 확인 필요")

print("Mask shape:", mask.shape)
print("Unique values:", len(np.unique(mask)))

thresholds = [0, 32, 64, 127, 192, 254]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, t in zip(axes.flat, thresholds):
    binary = mask > t

    overlay = rgb.copy()
    overlay[binary] = (
        rgb[binary] * 0.5
        + np.array([255, 0, 0]) * 0.5
    ).astype(np.uint8)

    ax.imshow(overlay)
    ax.set_title(f"Threshold > {t} / Pixels: {binary.sum():,}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2

# 검증 후 확정한 값으로 수정
MASK_THRESHOLD = None

def load_binary_mask(mask_path, threshold):
    if threshold is None:
        raise ValueError("검증된 MASK_THRESHOLD를 먼저 지정하세요.")

    with Image.open(mask_path) as im:
        mask = np.asarray(im)

    if mask.ndim != 2:
        raise ValueError("2D 마스크가 아닙니다.")

    if not np.issubdtype(mask.dtype, np.integer):
        raise ValueError("마스크 픽셀 형식을 확인하세요.")

    binary = (mask > threshold).astype(np.uint8)

    return binary


def extract_bboxes(binary):
    binary = np.ascontiguousarray(binary, dtype=np.uint8)

    if binary.ndim != 2:
        raise ValueError("2D 바이너리 마스크가 필요합니다.")

    if not np.isin(np.unique(binary), [0, 1]).all():
        raise ValueError("바이너리 값이 아닙니다.")

    count, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            binary,
            connectivity=8
        )
    )

    boxes = []

    for i in range(1, count):
        x = int(stats[i, cv2.CC_STAT_LEFT])
        y = int(stats[i, cv2.CC_STAT_TOP])
        w = int(stats[i, cv2.CC_STAT_WIDTH])
        h = int(stats[i, cv2.CC_STAT_HEIGHT])

        boxes.append({
            "component_id": i,
            "x0": x,
            "y0": y,
            "x1": x + w,
            "y1": y + h,
            "width": w,
            "height": h,
            "area_pixels": int(stats[i, cv2.CC_STAT_AREA])
        })

    return boxes

In [ ]:
%run realiad_multiclass_bbox.py --data-dir notebook_results_real --threshold 127 --limit 0 --class-mode category_defect

In [ ]:
from pathlib import Path
import pandas as pd
import json

BBOX_DIR = Path("notebook_results_real/bbox_results_multiclass")

boxes = pd.read_csv(BBOX_DIR / "defect_bboxes.csv")
classes = json.loads((BBOX_DIR / "classes.json").read_text(encoding="utf-8"))

print("전체 클래스 수:", len(classes["names"]))
print("추출된 Bounding Box:", len(boxes))

display(
    boxes.groupby(["class_id", "class_name"])
    .agg(
        bbox_count=("component_id", "count"),
        image_count=("image_id", "nunique")
    )
    .reset_index()
    .sort_values("class_id")
)

In [ ]:
from pathlib import Path
import pandas as pd
import json

BBOX_DIR = Path("notebook_results_real/bbox_results_multiclass")

boxes = pd.read_csv(BBOX_DIR / "defect_bboxes.csv")
classes = json.loads((BBOX_DIR / "classes.json").read_text(encoding="utf-8"))

print("전체 클래스 수:", len(classes["names"]))
print("추출된 Bounding Box:", len(boxes))

display(
    boxes.groupby(["class_id", "class_name"])
    .agg(
        bbox_count=("component_id", "count"),
        image_count=("image_id", "nunique")
    )
    .reset_index()
    .sort_values("class_id")
)

### 내보내기 확인

- CSV: 원본 xyxy·정규화 xywh·제품/뷰 키·성분 면적·박스 면적
- COCO: split별 JSON, category_id=1=defect, images에는 확인된 빈 마스크 뷰도 포함
- YOLO: `yolo_labels/<split>/<image_id>.txt`, class_id=0=defect
- **이미지는 복사하지 않았습니다.** detection_images.csv의 image_path→yolo_stem 매핑에 따라
  학습용 이미지를 같은 stem으로 준비하는 단계는 과제입니다. 원본 이름에 바로 붙이면 짝이 안 맞습니다.
- 미제공·오류는 제외 사유를 저장하며 정상 빈 txt로 만들지 않습니다.

원본 GT 박스 생성과 실제 RGB 기반 탐지 추론은 서로 다른 단계입니다.

## 6. 입력 해상도 1024 / 512 / 256 비교

**완성된 분석:** 동일 RGB의 bilinear resize, 성분 ID 맵의 명시적인 pixel-center nearest,
원본 성분별 잔존 픽셀·소실·축소 후 박스 폭·높이, 입력 메모리 산술 비교.
배열은 RGB uint8이며 모델 입력 `[5,3,H,W]` float32로 바뀔 때의 **입력 텐서만** 계산합니다.
활성값·가중치·작업공간·GPU 피크 메모리나 추론 속도는 포함하지 않습니다.

| 정사각형 입력 | 1뷰 FP32 입력 MiB | 5뷰 FP32 입력 MiB |
|---|---:|---:|
|1024|12|60|
|512|3|15|
|256|0.75|3.75|

합성 데모 원본은 512×640이므로 1024는 업샘플입니다. 원본에 없는 디테일을 복원하지 않습니다.
비정사각 원본의 square resize는 종횡비를 바꿉니다. 실제 학습에서 letterbox를 선택하면
박스·mask 변환과 역변환을 따로 구현하고 이번 분석과 다른 정책임을 기록하세요.
모델 내부에서 다시 800 등으로 resize하면 256/512/1024 비교가 무효가 되므로 입력 hook으로 확인합니다.

In [ ]:
%run realiad_5view_information_loss.py --data-dir notebook_results_real --split train --limit-products 10 --preview-products 2

In [ ]:
%run realiad_5view_information_loss.py --data-dir notebook_results_real --split train --limit-products 0 --preview-products 5

In [ ]:
%run analyze_realiad_information_loss.py --data-dir notebook_results_real

In [ ]:
%run analyze_realiad_information_loss.py --data-dir notebook_results_real --max-categories 30 --top-n 50

In [ ]:
from pathlib import Path
import pandas as pd

RESULT_DIR = Path(
    "notebook_results_real/resolution_information_loss/analysis"
)

steps = pd.read_csv(
    RESULT_DIR / "stepwise_group_changes.csv"
)

display(
    steps[
        [
            "category", "view",
            "from_size", "to_size",
            "image_count",
            "mae_delta_median",
            "edge_recall_tol1_delta_median",
            "gradient_energy_retention_delta_median",
            "local_contrast_retention_delta_median"
        ]
    ].head(30)
)

In [ ]:
from IPython.display import display, Image

display(Image(
    filename=str(RESULT_DIR / "views_edge_recall_tol1.png")
))

display(Image(
    filename=str(RESULT_DIR / "views_local_contrast_retention.png")
))

In [ ]:
detail = pd.read_csv(
    RESULT_DIR / "category_view_detailed.csv"
)

display(
    detail[detail["size"] == 256][
        [
            "category", "view",
            "image_count", "product_count",
            "mae_median",
            "edge_recall_tol1_median",
            "local_contrast_retention_median"
        ]
    ].sort_values("edge_recall_tol1_median")
    .head(30)
)

# Real-IAD 5시점 이미지 밝기 증강 구현 및 검증 절차

## 1. 개발 목적

Real-IAD 데이터의 품목(Category), 불량 유형(Defect Type), 촬영 시점(C1~C5)에 따른 밝기 특성을 분석하고, 이를 바탕으로 **불량 특징을 보존하면서 조명 변화에 강건한 모델을 학습하기 위한 밝기 증강 파이프라인**을 구현한다.

기존 밝기 비교표의 535개 행은 독립적인 밝기 유형이 아니라 품목·불량 유형·촬영 시점별 비교 조합이다. 일부 품목에서는 정상과 불량의 전체 평균 밝기 차이가 크게 나타나므로, 모든 이미지에 동일한 밝기 정규화나 강한 증강을 적용하지 않는다.

## 2. 구현 순서

### STEP 1. 데이터 및 기존 분석 결과 불러오기

**목적:** 기존 분석 결과를 재사용하고, 학습 데이터의 누수를 방지한다.

**구현 작업**
1. `image_metadata_with_split.csv`를 불러온다.
2. 밝기 비교표의 `category`, `view`, `defect_type`, `ok_mean`, `ng_mean`, `mean_diff`, `distribution_overlap`, `defect_relative_brightness`를 확인한다.
3. 동일 제품의 C1~C5 이미지를 `product_key`로 묶는다.
4. Train/Validation/Test 분할을 유지한다.
5. 실제 이미지 경로와 결함 마스크 경로를 연결한다.

**완료 조건:** 제품 단위 분할이 유지되고, 5시점 이미지 및 결함 마스크를 정확히 조회할 수 있어야 한다.

### STEP 2. 품목·시점별 밝기 통계 재분석

**목적:** 증강 범위를 결정하기 위한 근거를 확보한다.

**구현 작업**
1. Train 이미지의 평균 밝기, 표준편차, P05·P50·P95를 계산한다.
2. 품목 × 시점별 밝기 히스토그램을 생성한다.
3. 정상과 불량의 밝기 차이를 비교한다.
4. `mean_diff`와 `distribution_overlap`을 기준으로 검수 우선순위를 정한다.
5. 밝기 차이가 큰 조합의 실제 이미지를 확인한다.

**출력:** `train_brightness_statistics.csv`, `brightness_distribution.png`, `brightness_review_candidates.csv`

**주의:** 기존 535개 조합의 요약표만으로는 개별 이미지의 P05·P95를 복원할 수 없으므로, Train 원본 JPG에서 계산해야 한다.

### STEP 3. 밝기 증강 함수 구현

**목적:** OpenCV를 사용해 다양한 조명 조건을 재현한다.

| 증강 방법 | 초기 실험 범위 | OpenCV 구현 |
|---|---|---|
| 밝기·대비 변환 | α = 0.9~1.1, β = −10~10 | `cv2.convertScaleAbs` 대신 부호 보존형 float 연산 후 clip |
| Gamma Correction | γ = 0.9~1.1 | NumPy LUT + `cv2.LUT` |
| CLAHE | clipLimit = 1.5~2.0 | `cv2.createCLAHE` |
| 원본 유지 | 증강 없음 | Baseline |

**구현 작업**
1. 밝기·대비 변환 함수를 구현한다.
2. Gamma Correction 함수를 구현한다.
3. CLAHE를 별도 실험군으로 구현한다.
4. 각 증강 방법의 적용 확률을 설정한다.
5. 난수 시드를 고정하여 실험을 재현할 수 있도록 한다.
6. 과도한 밝기 포화 및 어두운 영역의 정보 소실을 검사한다.

**주의:** CLAHE는 단순 밝기 변화가 아니라 국소 대비까지 변경하므로 다른 밝기 증강과 구분하여 평가한다. 모든 초기 범위는 실험 후보이며 확정된 최적값이 아니다.

### STEP 4. 품목·시점별 적응형 증강 정책 구현

**목적:** 이미지 특성에 따라 증강 강도를 조절한다.

**구현 작업**
1. Train 밝기 통계를 기반으로 품목·시점별 허용 증강 범위를 설정한다.
2. 밝기 차이가 큰 정상·불량 조합은 우선 검수 대상으로 지정한다.
3. 정상과 불량 모두에 동일한 증강 정책을 적용하되, 불량 유형별로 별도 강도가 필요한지 실험한다.
4. 밝기 변화가 결함 자체를 제거하거나 새로운 결함처럼 보이는 현상을 방지한다.
5. 품목·시점별 증강 파라미터를 JSON 파일로 관리한다.

**출력:** `augmentation_policy.json`

**주의:** 정상·불량의 평균 밝기 차이가 크다는 이유만으로 그 차이를 제거해서는 안 된다. 해당 차이가 결함의 특징인지 촬영 조건의 편향인지 먼저 검증한다.

### STEP 5. 5시점 증강 정책 비교

**목적:** 실제 다중 카메라 환경을 반영한다.

**구현 작업**
1. 동일 제품의 C1~C5 이미지에 동일한 밝기 변화를 적용하는 공통 증강을 구현한다.
2. 시점마다 다른 밝기 변화를 적용하는 독립 증강을 구현한다.
3. 공통 변화와 독립 변화를 혼합한 증강을 구현한다.
4. 각 방법의 증강 전후 5시점 이미지를 비교한다.
5. 5시점 이미지의 제품 ID와 순서를 유지한다.

**실험군**
- A: 원본 유지
- B: 5시점 공통 밝기 증강
- C: 시점별 독립 밝기 증강
- D: 공통 + 독립 밝기 증강

**출력:** `five_view_augmentation_comparison.png`

### STEP 6. 결함 특징 보존 검증

**목적:** 밝기 증강으로 실제 불량 정보가 훼손되는지 확인한다.

**구현 작업**
1. 검증된 결함 마스크를 이용해 결함 ROI를 추출한다.
2. 증강 전후 결함 영역의 평균 밝기와 주변 영역 대비를 측정한다.
3. Canny 경계와 Laplacian 에너지 변화를 계산한다.
4. 포화 픽셀 비율과 암부 픽셀 비율을 측정한다.
5. 결함 영역의 상대 밝기 부호가 역전되는 사례를 확인한다.
6. 변화가 큰 이미지의 원본·증강 결과를 나란히 시각화한다.

**출력:** `defect_preservation_metrics.csv`, `defect_preservation_review.png`

**완료 조건:** 증강 후 결함 형태와 판별에 필요한 시각적 특징이 유지되는지 검수할 수 있어야 한다.

### STEP 7. 증강 정책별 모델 학습 실험

**목적:** 실제 검출 성능 개선 여부를 확인한다.

| 실험 | 학습 데이터 | 비교 목적 |
|---|---|---|
| E0 | 원본 | 기준 성능 |
| E1 | 밝기 정규화 | 밝기 편차 보정 효과 |
| E2 | 제한적 밝기 증강 | 조명 강건성 |
| E3 | 정규화 + 증강 | 병행 효과 |
| E4 | 품목·시점별 적응형 증강 | 데이터 특성 반영 효과 |

**구현 작업**
1. 모든 실험에 동일한 모델과 제품 단위 데이터 분할을 사용한다.
2. 학습 Epoch, Batch Size, 입력 해상도 및 평가 조건을 통일한다.
3. 학습 데이터에만 확률적 증강을 적용한다.
4. Validation 데이터는 증강하지 않고 평가한다.
5. NG Recall, FPR, F1-score 및 위치 탐지 성능을 기록한다.
6. 불량 유형별·품목별·시점별 성능 차이를 비교한다.

**출력:** `augmentation_experiment_results.csv`

### STEP 8. 최종 정책 선정 및 보고서 작성

**목적:** 분석 결과에 근거하여 최종 밝기 증강 정책을 선정한다.

**구현 작업**
1. E0~E4 실험 결과를 비교한다.
2. 불량 미검출(FN)이 증가한 실험을 분석한다.
3. 정상 제품의 오탐(FP)이 증가한 실험을 분석한다.
4. 품목·불량 유형·촬영 시점별 성능 변화를 확인한다.
5. 결함 특징 보존 검증과 Validation 성능을 종합하여 최종 정책을 선정한다.
6. Test 데이터로 최종 성능을 1회 평가한다.
7. 증강 설정과 실험 결과를 Markdown 보고서로 작성한다.

**최종 산출물**
- `augmentation_policy.json`
- `brightness_augmentation.py`
- `augmentation_validation.ipynb`
- `augmentation_experiment_results.csv`
- `brightness_augmentation_report.md`

## 3. 최종 구현 원칙

- **OpenCV 중심 구현:** 밝기 변환, Gamma, CLAHE, 결함 영역 검증을 OpenCV와 NumPy로 수행한다.
- **제품 단위 분할 유지:** 동일 제품의 5시점 이미지가 서로 다른 데이터 분할에 들어가지 않도록 한다.
- **결함 정보 우선:** 밝기 분포를 균일하게 만드는 것보다 불량의 시각적 특징을 유지하는 것을 우선한다.
- **단계별 검증:** 증강 이미지의 시각적 품질 검수 후 모델 학습을 진행한다.
- **재현성 확보:** 증강 파라미터, 난수 시드, 실험 설정 및 평가 결과를 기록한다.
- **최종 판단 기준:** 밝기 통계 변화만으로 증강 성공을 판정하지 않고 실제 모델의 Validation 성능으로 결정한다.

In [ ]:
def student_brightness(rgb, *, rng, config):
    """TODO: RGB uint8 HWC -> RGB uint8 HWC, same shape, deterministic with rng."""
    raise NotImplementedError('과제 A: 밝기 증강을 직접 구현하세요.')

RUN_BRIGHTNESS_ASSIGNMENT = False
if RUN_BRIGHTNESS_ASSIGNMENT:
    train_source = selected[selected.project_split.eq('train') & selected.status.eq('ok')]
    if train_source.empty:
        raise ValueError('증강 소스에 사용할 train 이미지가 없습니다.')
    candidate = student_brightness(read_rgb(ROOT/train_source.iloc[0].image_path),
        rng=np.random.default_rng(SEED),config={})
    # TODO: 원본/증강, 밝기 분포, 포화율·결함 대비와 동일 좌표 유지 검증


### 과제 B. OpenCV 기하·광학 증강과 라벨 동기화

1. 학습 카메라 의미를 유지하는 flip·작은 rotate·translate 후보를 고릅니다.
2. 기하 변환 행렬과 난수를 **한 번** 뽑고 RGB·mask에 동일하게 적용합니다.
3. RGB는 적절한 연속 보간, mask는 nearest, 경계 채움 정책을 명시합니다.
4. 변환된 mask에서 `mask_to_boxes`로 박스를 **다시 추출**합니다.
5. 잘려서 사라진 결함은 빈 라벨로 처리하되, 해당 합성 뷰 라벨·제품 묶음 관계도 검토합니다.
6. blur·noise·contrast 등 광학 증강은 불량 흔적이 지워지지 않는 강도를 검수합니다.

힌트 API: `cv2.flip`, `cv2.getRotationMatrix2D`, `cv2.warpAffine`, `cv2.GaussianBlur`, `cv2.LUT`.
직접 박스를 변환한다면 네 모서리를 변환한 뒤 범위를 제한하세요. 회전 시 두 모서리만 옮기면 틀릴 수 있습니다.
가짜 불량을 만든 광학 잡음과 기존 실제 결함 GT가 맞는지 검수하세요.
**다른 제품의 5뷰를 섞지 말고**, augmentation seed와 source_sample_key를 기록하세요.

In [ ]:
def student_opencv_augment(rgb, mask, *, rng, config):
    """TODO: return augmented_rgb, augmented_binary_mask, transform_metadata.
    광학 변환만이면 mask를 유지. 기하 변환이면 같은 변환을 적용.
    """
    raise NotImplementedError('과제 B: OpenCV 증강과 마스크 동기화를 구현하세요.')

RUN_OPENCV_ASSIGNMENT = False
if RUN_OPENCV_ASSIGNMENT:
    import cv2  # requirements-assignment.txt 설치 후 사용
    if positive.empty:
        raise ValueError('train 양성 마스크 제품부터 준비하세요.')
    train_positive=positive[positive.project_split.eq('train')]
    if train_positive.empty:
        raise ValueError('과제 증강 소스에 사용할 train 양성 마스크가 없습니다.')
    rec=train_positive.iloc[0]
    x=read_rgb(ROOT/rec.image_path);m=read_mask(ROOT/rec.mask_path,x.shape[:2])
    aug_x,aug_m,transform=student_opencv_augment(x,m,rng=np.random.default_rng(SEED),config={})
    aug_boxes=mask_to_boxes(aug_m)
    # TODO: dtype/shape/결함 보존/원본 분할 계보/박스 오버레이 검증 후 승인


### 과제 C. 1024/512/256 모델 입력 파이프라인

1. 같은 원본 제품 분할을 고정하고 각 크기의 resize/letterbox 방식을 선언합니다.
2. RGB·mask·박스를 동기 변환하고 변환 메타데이터를 남깁니다.
3. RGB HWC uint8 → 모델 요구 CHW float·정규화. 탐지기의 내장 전처리와 중복 적용하지 않습니다.
4. 배치 직전과 실제 백본 입력 직전 크기를 확인합니다.
5. 입력 크기·최대 epoch/step·유효 batch·사전학습 가중치·seed를 기록합니다.
6. 학습 시간·피크 메모리·제품 Recall·박스 AP50/AP@[.50:.95]·작은 불량 Recall을 비교합니다.

힌트: 메모리 제한으로 1024 batch가 줄어들면 accumulation과 유효 batch를 기록하세요.
작은 불량 mask 소실률을 모델 탐지 Recall로 대신 쓰면 안 됩니다.
검증/test는 고정된 결정적 전처리만 적용하고 **랜덤 증강은 train에만** 적용합니다.

In [ ]:
def student_prepare_input(rgb, mask, *, target_size, config):
    """TODO: model input + transformed mask + boxes + transform metadata.
    1024/512/256의 실제 네트워크 입력 크기까지 확인할 것.
    """
    raise NotImplementedError('과제 C: 모델 입력 전처리를 구현하세요.')


### 과제 D. 증강 후 실제 학습·평가

공개 사전학습 가중치가 있는 탐지기 **하나**를 먼저 선택해 데이터 파이프라인을 검증합니다.
모델 12종 전체 비교는 프로젝트 후속 단계이며 이번 과제에서 한꺼번에 수행하지 않습니다.

| 단계 | 해야 할 작업 | 통과 기준 |
|---|---|---|
|D1|COCO 또는 YOLO dataset 구성, 원본→라벨 ID 연결|빈 라벨·다중 박스·잘린 결함 확인|
|D2|사전학습 모델 로드·불량 클래스 헤드 설정|학습 가능한 파라미터·로드 제외 키 기록|
|D3|소규모 train 제품으로 과적합 확인|loss 감소·박스가 결함 위치를 덮음|
|D4|고정 split에서 baseline 학습|validation 선택, test 미사용|
|D5|동일 조건으로 밝기/OpenCV/결합 증강 비교|증강 수·노출 step·seed·유효 batch 기록|
|D6|상위 증강 정책의 1024/512/256 비교|실제 입력 크기·성능·비용 비교|
|D7|validation에서 정책 확정, test 최종 평가|반복 test 튜닝 금지|

**권장 실험 수:** 512에서 증강 4조건 + 선정 정책의 1024·256 추가 = 총 6조건.
여유가 있으면 3해상도×4증강=12조건으로 확대합니다.
내장 Mosaic·HSV 등은 baseline에서 끄거나 전 조건에 동일 적용해 추가 OpenCV 효과와 구분하세요.
제품 판정은 5뷰 관찰을 모아 평가하고 이미지 단위 지표와 구분합니다.

In [ ]:
AUGMENTATIONS = ('none','brightness','opencv','brightness_opencv')
training_plan = pd.DataFrame([
    {'experiment_id':f'e{i+1:02d}','input_size':512,'augmentation':policy,
     'seed':SEED,'split_hash':'TODO','model_id':'TODO','pretrained_id':'TODO',
     'effective_batch':'TODO','max_steps':'TODO','status':'NOT_RUN'}
    for i,policy in enumerate(AUGMENTATIONS)])
# 선정 정책을 정한 뒤 1024·256 행을 추가하세요.
training_results = pd.DataFrame(columns=['experiment_id','actual_input_size','n_train_products',
    'product_recall','product_fpr','box_ap50','box_map_50_95','small_defect_recall',
    'five_view_p95_ms','peak_memory_mib','train_seconds','status'])
display(training_plan);display(training_results)
training_plan.to_csv(OUTPUT/'training_plan_TEMPLATE.csv',index=False)
training_results.to_csv(OUTPUT/'training_results_EMPTY.csv',index=False)

def student_train_and_evaluate(experiment, *, train_manifest, validation_manifest):
    """TODO: model init, dataloader, train, checkpoint selection, held-out validation.
    Return measured results only; test is not passed to model selection.
    """
    raise NotImplementedError('과제 D: 학습·검증 루프와 지표를 직접 구현하세요.')

RUN_TRAINING_ASSIGNMENT = False
if RUN_TRAINING_ASSIGNMENT:
    raise NotImplementedError('먼저 train/val 데이터로더·학습·평가·내장증강 설정을 연결하세요.')


## 8. 제출물·평가 기준·실패 사례

- **필수 분석:** 클래스별 밝기 표·그림·해석, train-only 히트맵+분모+GAN 위치 제안,
  bbox CSV/오버레이/라벨 매핑, 해상도별 소실·박스 크기·RGB·입력 메모리 비교
- **과제:** 밝기 함수, OpenCV 동기 증강 함수, 모델 입력 변환 함수, 학습·검증 루프,
  train 전용 증강과 동일 원본 분할의 증거, 실제 측정 결과표, 실패 사례
- **점수 예시:** 분석 40점(밝기/히트맵/박스/해상도 각 10), 증강·라벨 정합성 25,
  학습·공정 비교 25, 재현·해석 10. 수업에 맞게 조정 가능
- **필수 실패 확인:** overflow, RGB/BGR 혼동, mask bilinear 보간, 미제공 mask를 정상화,
  test 누수, 다른 뷰 좌표 병합, 박스 원본 좌표 불일치, 모델 내부 강제 resize
- 결과 숫자를 채우기 위해 임의 점수·학습 성공·가짜 지연을 만들지 않습니다.
  실패·미실행은 그대로 기록하고 실제 데이터 학습과 합성 데모 분석을 구분하세요.

### 두 구현 교차 확인
`core.py`의 기존 비교기는 일반 분석용입니다. 이번 네 분석은 `compare_focused.py`로 별도 비교합니다.
두 버전에서 동일 selected_manifest·split·크기·grid 설정을 써야 합니다.
시간 값은 비교 대상에서 제외하며 수학 결과·라벨·heatmap 배열을 대조합니다.

In [ ]:
print('analysis output:', RESULTS.resolve())
print('현재 실행은 분석 데모입니다. 증강·학습 과제의 status는 NOT_RUN입니다.')
print('교차 비교 예: python compare_focused.py <OpenCV focused_analysis> <ScikitImage focused_analysis>')


## 참고 API

- OpenCV 성분·박스: https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html
- OpenCV dtype·픽셀 연산: https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html
- OpenCV 기하 변환·보간: https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html
- scikit-image label·regionprops: https://scikit-image.org/docs/stable/api/skimage.measure.html
- 실제 마스크 값·카메라 매핑은 내려받은 Real-IAD 릴리스에서 먼저 검증합니다.
